In [ ]:
from transformers import RobertaTokenizer, RobertaForMaskedLM
import torch

In [ ]:
model_name = 'roberta-base'
tokenizer = RobertaTokenizer.from_pretrained(model_name)
model = RobertaForMaskedLM.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

In [ ]:
text = "Uma rede neural é um tipo de <mask> que é usado para processamento de linguagem natural."

In [ ]:
inputs = tokenizer(text, return_tensors='pt')
print(inputs)

{'input_ids': tensor([[    0,   791,  1916, 35117, 26739,  7935,  7252,  4767,   139,   263,
         50264,  1192,  7935,   201,  2102,  3840,   609,  5511,   139,   263,
         38954,  1073,   991,  1632,     4,     2]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1]])}


In [ ]:
with torch.no_grad():
  outputs = model(**inputs)
print(outputs)

MaskedLMOutput(loss=None, logits=tensor([[[33.0944, -3.7791, 17.9557,  ...,  1.6636,  5.2148, 10.5883],
         [ 6.6143, -2.2842, 16.7856,  ..., -1.4340,  1.5761,  5.9353],
         [ 2.6541, -2.8336, 10.2844,  ..., -3.5465, -0.5512,  1.2722],
         ...,
         [-1.0966, -4.5988,  6.0705,  ..., -4.6206, -6.1500,  1.0818],
         [18.8709, -4.4813, 19.1379,  ..., -0.6549,  2.7602,  6.5704],
         [10.9910, -3.7715, 28.3436,  ..., -1.9791, -4.2113,  8.4504]]]), hidden_states=None, attentions=None)


In [ ]:
mask_token_index = torch.where(inputs['input_ids'] == tokenizer.mask_token_id)[1]
print(mask_token_index)

tensor([10])


In [ ]:
mask_token_logits = outputs.logits[0, mask_token_index, :]
print(mask_token_logits)

tensor([[-4.9231, -3.1930,  2.1638,  ..., -8.5636, -8.0436, -3.1378]])


In [ ]:
probabilities = torch.softmax(mask_token_logits, dim=-1)
print(probabilities)

tensor([[1.6821e-09, 9.4884e-09, 2.0119e-06,  ..., 4.4134e-11, 7.4233e-11,
         1.0028e-08]])


In [ ]:
top_5_tokens = torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()
print(top_5_tokens)

[2257, 414, 1468, 35277, 12769]


In [ ]:
predicted_words = [tokenizer.decode([token]).strip() for token in top_5_tokens]
print(predicted_words)

['software', 'data', 'material', 'arte', 'manual']


In [ ]:
print(f"Palavras preditas para a máscara: {predicted_words}")

Palavras preditas para a máscara: ['software', 'data', 'material', 'arte', 'manual']
